<a href="https://colab.research.google.com/github/hasanulazad-cloud/Project-Title---Data-Cleaning-Pipeline-for-SaaS-Dataset/blob/main/Copy_of_Python_Module_End_Assessment_1_Data_Cleaning_Pipeline_for_SaaS_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# TASK 1: INITIALIZE AND DISPLAY THE DATASET
# dataset
saas_data = {
    "customer_id": [101, 102, 103, 104, 105, 106, 107],
    "name": ["Rahul Sharma", "Meena Nair", None, "A. Kumar", "David", "Priya", "Rahul Sharma"],
    "signup_date": ["2023-01-10", "2023/02/15", "03-01-2023", "15-04-2023", "2023-05-10", "20230520", "2023-01-10"],
    "plan": ["Pro", "Basic", "Pro", "Enterprise", "Pro", "", "Pro"],
    "monthly_fee": ["49.99", "19.99", "forty-nine", "99.99", "49.99", "49.99", "49.99"],
    "usage_hours": ["120", "45", "88", None, "NULL", "200", "120"]
}

# Print each record using index loop
for i in range(len(saas_data["customer_id"])):
    print(
        f"ID: {saas_data['customer_id'][i]} | "
        f"Name: {saas_data['name'][i]} | "
        f"Signup Date: {saas_data['signup_date'][i]} | "
        f"Plan: {saas_data['plan'][i]} | "
        f"Monthly Fee: {saas_data['monthly_fee'][i]} | "
        f"Usage Hours: {saas_data['usage_hours'][i]}"
    )

ID: 101 | Name: Rahul Sharma | Signup Date: 2023-01-10 | Plan: Pro | Monthly Fee: 49.99 | Usage Hours: 120
ID: 102 | Name: Meena Nair | Signup Date: 2023/02/15 | Plan: Basic | Monthly Fee: 19.99 | Usage Hours: 45
ID: 103 | Name: None | Signup Date: 03-01-2023 | Plan: Pro | Monthly Fee: forty-nine | Usage Hours: 88
ID: 104 | Name: A. Kumar | Signup Date: 15-04-2023 | Plan: Enterprise | Monthly Fee: 99.99 | Usage Hours: None
ID: 105 | Name: David | Signup Date: 2023-05-10 | Plan: Pro | Monthly Fee: 49.99 | Usage Hours: NULL
ID: 106 | Name: Priya | Signup Date: 20230520 | Plan:  | Monthly Fee: 49.99 | Usage Hours: 200
ID: 107 | Name: Rahul Sharma | Signup Date: 2023-01-10 | Plan: Pro | Monthly Fee: 49.99 | Usage Hours: 120


In [2]:
# TASK 2: ADD NEW CUSTOMER RECORDS

# Need to add one customer
num_records_to_add = 1
print(f"How many customers do you want to add? {num_records_to_add}")

# please run below the provided example input
simulated_names = ["Arun Kumar"]
simulated_dates = ["2024-01-05"]
simulated_plans = ["Basic"]
simulated_fees = ["19.99"]
simulated_hours = ["60"]

for _ in range(num_records_to_add):
    # Find the maximum ID currently in our list and add 1
    new_id = max(saas_data["customer_id"]) + 1

    # Take the values from our lists
    name_input = simulated_names[0]
    date_input = simulated_dates[0]
    plan_input = simulated_plans[0]
    fee_input = simulated_fees[0]
    hours_input = simulated_hours[0]

    print(f"Enter name: {name_input}")
    print(f"Enter signup date: {date_input}")
    print(f"Enter plan: {plan_input}")
    print(f"Enter monthly fee: {fee_input}")
    print(f"Enter usage hours: {hours_input}")

    # Append the values directly into our lists
    saas_data["customer_id"].append(new_id)
    saas_data["name"].append(name_input)
    saas_data["signup_date"].append(date_input)
    saas_data["plan"].append(plan_input)
    saas_data["monthly_fee"].append(fee_input)
    saas_data["usage_hours"].append(hours_input)

print("Record successfully added!\n")

How many customers do you want to add? 1
Enter name: Arun Kumar
Enter signup date: 2024-01-05
Enter plan: Basic
Enter monthly fee: 19.99
Enter usage hours: 60
Record successfully added!



In [3]:
import re

# TASK 3: THE SaaSCustomer CLASS (VERY SIMPLE VERSION)

class SaaSCustomer:
    def __init__(self, customer_id, name, signup_date, plan, monthly_fee, usage_hours):
        self.customer_id = customer_id

        # Clean name: Set to 'Unknown Customer' if None or empty string
        self.name = "Unknown Customer" if name is None or str(name).strip() == "" else name

        self.signup_date = signup_date # Assuming date cleaning is handled elsewhere for now

        # Clean plan: Convert to title case
        self.plan = str(plan).strip().capitalize() if plan else "Unknown"

        # Clean monthly_fee: Convert to float, if conversion fails, set to None
        try:
            self.monthly_fee = float(monthly_fee)
        except (ValueError, TypeError):
            self.monthly_fee = None

        # Clean usage_hours: Convert to int, if conversion fails, set to 0
        try:
            self.usage_hours = int(usage_hours)
        except (ValueError, TypeError):
            self.usage_hours = 0

    def display_customer(self):
        print(f"Customer ID: {self.customer_id}, Name: {self.name}, Signup Date: {self.signup_date}, Plan: {self.plan}, Fee: {self.monthly_fee}, Hours: {self.usage_hours}")


# Let's create a test customer using the problematic row 103 data:

# It has a missing name (None), lowercase plan ("pro"), and text fee ("forty-nine")
test_customer = SaaSCustomer(
    customer_id=103,
    name=None,
    signup_date="03-01-2023",
    plan="pro",
    monthly_fee="forty-nine",
    usage_hours="88"
)

# Call the display method to see if the name fix worked
test_customer.display_customer()

# Print out the hidden attributes to see how nicely they were cleaned up!
print(f"Cleaned Plan: {test_customer.plan}")
print(f"Cleaned Fee:  {test_customer.monthly_fee}")
print(f"Cleaned Hours: {test_customer.usage_hours}")

Customer ID: 103, Name: Unknown Customer, Signup Date: 03-01-2023, Plan: Pro, Fee: None, Hours: 88
Cleaned Plan: Pro
Cleaned Fee:  None
Cleaned Hours: 88


In [4]:
#Task 4 – Data Cleaning Pipeline
from datetime import datetime
# 1. CLEANING FUNCTION: HANDLE MISSING VALUES

def handle_missing_values(data):
    # Create a fresh, empty dictionary to hold the cleaned rows
    cleaned_data = {
        "customer_id": [], "name": [], "signup_date": [],
        "plan": [], "monthly_fee": [], "usage_hours": []
    }

    total_items = len(data["customer_id"])

    for i in range(total_items):
        # --- Rule A: Fix Name ---
        current_name = data["name"][i]
        if current_name == None or str(current_name).strip() == "":
            current_name = "Unknown Customer."

        # --- Rule B: Fix Usage Hours ---
        current_hours = data["usage_hours"][i]
        if current_hours == None or str(current_hours).upper() in ["NULL", "NONE", ""]:
            current_hours = 0
        else:
            try:
                current_hours = int(current_hours)
            except (ValueError, TypeError):
            # Added TypeError for robustness
                current_hours = 0

        # --- Rule C: Fix Monthly Fee ---
        current_fee = data["monthly_fee"][i]
        try:
            current_fee = float(current_fee)
        except (ValueError, TypeError):
          # Added TypeError to handle cases where fee is already None
            current_fee = None
            # Text like 'forty-nine' safely becomes None

        # Add all our cleaned values into our fresh dictionary
        cleaned_data["customer_id"].append(data["customer_id"][i])
        cleaned_data["name"].append(current_name)
        cleaned_data["signup_date"].append(data["signup_date"][i])
        cleaned_data["plan"].append(data["plan"][i])
        cleaned_data["monthly_fee"].append(current_fee)
        cleaned_data["usage_hours"].append(current_hours)

    return cleaned_data

# 2. CLEANING FUNCTION: REMOVE DUPLICATE RECORDS

def remove_duplicates(data):
    cleaned_data = {
        "customer_id": [], "name": [], "signup_date": [],
        "plan": [], "monthly_fee": [], "usage_hours": []
    }

    # An empty list to keep track of customer IDs we have already processed
    seen_ids = []

    total_items = len(data["customer_id"])

    for i in range(total_items):
        current_id = data["customer_id"][i]

        # If this ID is NOT in our seen_ids list, it means it's unique!
        if current_id not in seen_ids:
            seen_ids.append(current_id)
            # Remember this ID so we don't copy it again

            # Copy all data fields for this non-duplicate row
            cleaned_data["customer_id"].append(data["customer_id"][i])
            cleaned_data["name"].append(data["name"][i])
            cleaned_data["signup_date"].append(data["signup_date"][i])
            cleaned_data["plan"].append(data["plan"][i])
            cleaned_data["monthly_fee"].append(data["monthly_fee"][i])
            cleaned_data["usage_hours"].append(data["usage_hours"][i])

    return cleaned_data

# 3. CLEANING FUNCTION: STANDARDIZE DATE FORMATS

def standardize_dates(data):
    total_items = len(data["signup_date"])

    # List of different date layouts that exist in our messy data
    possible_formats = ["%Y-%m-%d", "%Y/%m/%d", "%d-%m-%Y", "%Y%m%d"]

    for i in range(total_items):
        date_text = str(data["signup_date"][i]).strip()
        converted_date = "Invalid Date"

        # Try matching the messy date against each template row by row
        for pattern in possible_formats:
            try:
                # If Python successfully reads the template layout:
                parsed_object = datetime.strptime(date_text, pattern)
                # Reformat it directly to standard YYYY-MM-DD
                converted_date = parsed_object.strftime("%Y-%m-%d")
                break # Stop testing templates because we found our match!
            except ValueError:
                continue
                # If it fails, silently move to the next pattern template

        # Update the list with our freshly standardized date string
        data["signup_date"][i] = converted_date

    return data

# Step 1: Fix missing/bad values first
step1_data = handle_missing_values(saas_data)

# Step 2: Pass Step 1's results into the duplicate remover
step2_data = remove_duplicates(step1_data)

# Step 3: Pass Step 2's results into the date fixer to get the final clean data
final_cleaned_data = standardize_dates(step2_data)

print("Data Cleaning Complete!\n")

# Let's print out the results to verify everything looks great!
for i in range(len(final_cleaned_data["customer_id"])):
    print(f"ID: {final_cleaned_data['customer_id'][i]} | Date: {final_cleaned_data['signup_date'][i]} | Fee: {final_cleaned_data['monthly_fee'][i]} | Hours: {final_cleaned_data['usage_hours'][i]}")

Data Cleaning Complete!

ID: 101 | Date: 2023-01-10 | Fee: 49.99 | Hours: 120
ID: 102 | Date: 2023-02-15 | Fee: 19.99 | Hours: 45
ID: 103 | Date: 2023-01-03 | Fee: None | Hours: 88
ID: 104 | Date: 2023-04-15 | Fee: 99.99 | Hours: 0
ID: 105 | Date: 2023-05-10 | Fee: 49.99 | Hours: 0
ID: 106 | Date: 2023-05-20 | Fee: 49.99 | Hours: 200
ID: 107 | Date: 2023-01-10 | Fee: 49.99 | Hours: 120
ID: 108 | Date: 2024-01-05 | Fee: 19.99 | Hours: 60


In [5]:
# TASK 5: COMPUTE METRICS AND SUMMARY STATISTICS

total_customers = len(final_cleaned_data["customer_id"])
total_fee_sum = 0.0
valid_fee_count = 0
total_usage_hours = 0
plan_counts = {}

for i in range(total_customers):
    fee = final_cleaned_data["monthly_fee"][i]
    if fee is not None:
        total_fee_sum += fee
        valid_fee_count += 1

    total_usage_hours += final_cleaned_data["usage_hours"][i]

    # Process plan counts
    p_name = str(final_cleaned_data["plan"][i]).strip().capitalize()
    if p_name == "":
        p_name = "Unknown"

    if p_name in plan_counts:
        plan_counts[p_name] += 1
    else:
        plan_counts[p_name] = 1

average_monthly_fee = total_fee_sum / valid_fee_count if valid_fee_count > 0 else 0.0

# Print final outputs matching requested format
print(f"Total Customers: {total_customers}")
print(f"Average Monthly Fee: {round(average_monthly_fee, 2)}")
print(f"Total Usage Hours: {total_usage_hours}")
print("\nPlan Distribution")
for p_name, count in plan_counts.items():
    print(f"{p_name} - {count}")
print()

Total Customers: 8
Average Monthly Fee: 48.56
Total Usage Hours: 633

Plan Distribution
Pro - 4
Basic - 2
Enterprise - 1
Unknown - 1



In [6]:
# TASK 6: IDENTIFY LOW-USAGE CUSTOMERS

def find_low_usage_customers(data):
    print("Low Usage Customers")
    for i in range(len(data["customer_id"])):
        hours = data["usage_hours"][i]
        if hours is not None and hours < 50:
            print(f"ID: {data['customer_id'][i]} | Name: {data['name'][i]} | Plan: {data['plan'][i]} | Usage Hours: {hours}")

find_low_usage_customers(final_cleaned_data)

Low Usage Customers
ID: 102 | Name: Meena Nair | Plan: Basic | Usage Hours: 45
ID: 104 | Name: A. Kumar | Plan: Enterprise | Usage Hours: 0
ID: 105 | Name: David | Plan: Pro | Usage Hours: 0


In [7]:
# TASK 7: UNIQUE PLAN TYPES IN ALPHABETICAL ORDER

unique_plans_set = set()
for i in range(len(final_cleaned_data["plan"])):
    plan_item = str(final_cleaned_data["plan"][i]).strip().capitalize()
    if plan_item == "":
        plan_item = "Unknown"
    unique_plans_set.add(plan_item)

sorted_plans_list = list(unique_plans_set)
sorted_plans_list.sort()

print("Available Plans")
print(sorted_plans_list)
print()

Available Plans
['Basic', 'Enterprise', 'Pro', 'Unknown']



In [8]:
# TASK 8: SAVE DATASETS OUT TO EXTERNAL FILES (ERROR HANDLING)

# Write out to saas_customers_cleaned.csv
try:
    csv_file = open("saas_customers_cleaned.csv", "w", encoding="utf-8")
    for i in range(len(final_cleaned_data["customer_id"])):
        fee_val = final_cleaned_data["monthly_fee"][i]
        fee_str = str(fee_val) if fee_val is not None else ""

        row_line = (
            f"{final_cleaned_data['customer_id'][i]}, {final_cleaned_data['name'][i]}, "
            f"{final_cleaned_data['signup_date'][i]}, {final_cleaned_data['plan'][i]}, "
            f"{fee_str}, {final_cleaned_data['usage_hours'][i]}\n"
        )
        csv_file.write(row_line)
except IOError as err:
    print(f"Error writing CSV file: {err}")
else:
    print("Success: 'saas_customers_cleaned.csv' has been generated.")
finally:
    try:
        csv_file.close()
    except NameError:
        pass

# Write out to plan_summary.txt
try:
    txt_file = open("plan_summary.txt", "w", encoding="utf-8")
    for plan, count in plan_counts.items():
        txt_file.write(f"{plan}: {count}\n")
except IOError as err:
    print(f" Error writing Text file: {err}")
else:
    print(" Success: 'plan_summary.txt' has been generated.")
finally:
    try:
        txt_file.close()
    except NameError:
        pass


Success: 'saas_customers_cleaned.csv' has been generated.
 Success: 'plan_summary.txt' has been generated.
